
Limpieza del dataset - Encuesta Distrital de Percepción y Cultura Ciudadana
Eje: Violencia contra la mujer
DataJam - filtrado y limpieza inicial


In [2]:
import pandas as pd
import geopandas as gpd
import seaborn as sb
import numpy as np
from google.colab import drive
import os
import chardet
import csv
import io
import matplotlib.pyplot as plt

# 1. CONFIGURACIÓN


In [3]:
drive.mount('/content/drive')

RUTA_ENTRADA = "/content/drive/MyDrive/base_ano_movil_2025.csv"
RUTA_SALIDA = "base_violencia_mujer_limpia.csv"

COLUMNAS_VIOLENCIA_MUJER = [
    # Llaves / identificación
    "DIRECTORIO_MZ", "DIRECTORIO_PRED", "DIRECTORIO_HOG", "DIRECTORIO_PER",

    # Llaves geográficas y temporales (cruce)
    "periodo", "SECTOR", "Cod_Locali", "Nom_Locali", "Cod_UPL", "Nom_UPL",

    # Perfil sociodemográfico
    "A6x2", "A6x3", "C1", "D1", "E1", "E1x1", "G1", "H1",
    "C303", "sexo_jefe",

    # Roles de género / distribución de tareas domésticas
    "Ax201", "Bx201", "Cx201", "Dx201", "Ex201", "Fx201",
    "Gx201", "Hx201", "Ix201", "Jx201",
    "Ax202", "ind_distribuciontareas_202",

    # Violencia intrafamiliar (delito sufrido)
    "Jx402", "Jx403",

    # Acoso sexual, violencia intrafamiliar y contra la mujer presenciados (por lugar)
    "Kx404_1", "Kx404_2", "Kx404_3", "Kx404_4", "Kx404_5", "Kx404_6",
    "Lx404_1", "Lx404_2", "Lx404_3", "Lx404_4", "Lx404_5", "Lx404_6",
    "Mx404_1", "Mx404_2", "Mx404_3", "Mx404_4", "Mx404_5", "Mx404_6",
    "Nx404_1", "Nx404_2", "Nx404_3", "Nx404_4", "Nx404_5", "Nx404_6",

    # Percepción de seguridad
    "F405", "G406", "IPS_dia", "IPS_noche",
    "IPSJ_A", "IPSJ_C", "IPSJ_E",

    # Inclusión, diversidad, confianza
    "Dx704", "ICG_D", "Bx704", "ICG_B",

    # Salud mental (GAD-7)
    "Ax102", "Bx102", "Cx102", "Dx102", "Ex102", "Fx102", "Gx102",
    "A101", "ind_salud_101", "ind_salud_102",

    # Factores de expansión (para estimaciones representativas)
    "fexp_calp_anu", "fexp_calh_anu",
]


Mounted at /content/drive


# 2. CARGA DEL DATASET

In [4]:
print("Cargando dataset...")
df = pd.read_csv(RUTA_ENTRADA, encoding="utf-8", low_memory=False)

# Elimina columna índice sin nombre si viene del export original (";","")
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]

print(f"Dimensiones originales: {df.shape}")

Cargando dataset...
Dimensiones originales: (13082, 286)


# 3. VALIDAR QUE LAS COLUMNAS EXISTEN

In [5]:
faltantes = [c for c in COLUMNAS_VIOLENCIA_MUJER if c not in df.columns]
if faltantes:
    print("Advertencia: columnas no encontradas en el dataset:")
    print(faltantes)

columnas_disponibles = [c for c in COLUMNAS_VIOLENCIA_MUJER if c in df.columns]

# 4. FILTRAR COLUMNAS DE INTERÉS


In [6]:
df_filtrado = df[columnas_disponibles].copy()
print(f"Dimensiones tras filtrar columnas: {df_filtrado.shape}")

Dimensiones tras filtrar columnas: (13082, 81)


# 5. LIMPIEZA DE VALORES


In [7]:
# 5.1 Normalizar strings "NA", "N/A", "" a NaN reales
df_filtrado = df_filtrado.replace(
    to_replace=["NA", "N/A", "na", "n/a", "", "-", "999", "9999"],
    value=np.nan
)

# 5.2 Convertir columnas binarias (0/1) de presenciados (K,L,M,N x404) a numérico
cols_binarias = [
    c for c in columnas_disponibles
    if c.startswith(("Kx404", "Lx404", "Mx404", "Nx404"))
]
for col in cols_binarias:
    df_filtrado[col] = pd.to_numeric(df_filtrado[col], errors="coerce")

# 5.3 Convertir columnas de escala/indicadores a numérico
cols_indicadores = [
    c for c in columnas_disponibles
    if c.startswith(("IPS_", "IPSJ_", "ICG_", "ind_salud", "ind_distribuciontareas"))
    or c in ["F405", "G406", "A101"]
]
for col in cols_indicadores:
    df_filtrado[col] = pd.to_numeric(df_filtrado[col], errors="coerce")

# 5.4 Normalizar variables categóricas de sexo/género
if "D1" in df_filtrado.columns:
    df_filtrado["D1"] = df_filtrado["D1"].astype("Int64")

if "sexo_jefe" in df_filtrado.columns:
    df_filtrado["sexo_jefe"] = df_filtrado["sexo_jefe"].astype("Int64")

# 5.5 Normalizar periodo a formato consistente (año-mes)
if "periodo" in df_filtrado.columns:
    df_filtrado["periodo"] = df_filtrado["periodo"].astype(str).str.strip()

# 5.6 Normalizar nombres de localidad (mayúsculas/espacios)
if "Nom_Locali" in df_filtrado.columns:
    df_filtrado["Nom_Locali"] = (
        df_filtrado["Nom_Locali"].astype(str).str.strip().str.upper()
    )

9. GUARDAR DATASET LIMPIO

In [8]:
df_filtrado.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
print(f"\nDataset limpio guardado en: {RUTA_SALIDA}")
print(f"Dimensiones finales: {df_filtrado.shape}")


Dataset limpio guardado en: base_violencia_mujer_limpia.csv
Dimensiones finales: (13082, 81)


# 10. DESCARGAR EL CSV A TU COMPUTADOR

In [ ]:
from google.colab import files

files.download(RUTA_SALIDA)